# Appendix Full Results Table Generation
This notebook is for generating tables which show overall model performance across all experiments

## Setup

In [2]:
import matplotlib.pyplot as plt
import numpy as np 
import pandas as pd

In [3]:
# Move up to main repo directory
import os
os.chdir("../..") # ONLY RUN THIS ONCE

In [4]:
## Load CSV file from file path string

def load_csv(file_path):
    """Load a CSV file and return a pandas DataFrame."""
    try:
        df = pd.read_csv(file_path)
        print(f"Loaded CSV file: {file_path}")
        return df
    except Exception as e:
        print(f"Error loading CSV file: {e}")
        return None

In [5]:
rewards_per_scale = {
    'high_scale': [75,50,25],
    'low_scale': [0.75,0.5,0.25],
    'high_neg_scale': [-25, -50, -75],
    'low_neg_scale': [-0.25, -0.5, -0.75]
}

## Main Logic

In [6]:
## First, we load in the merged dataframe
eval_path = 'experiments/merged_eval_results.csv'
results_df = load_csv(eval_path)

# metrics = ['cum_regret', 'sem_optimal_actions', 'reward_optimal_actions']
metrics = ['cum_regret']

Loaded CSV file: experiments/merged_eval_results.csv


### Regret

In [47]:
from itertools import product
import numpy as np

# Standardize string columns to avoid matching issues
for col in ['Model', 'History', 'Variance', 'Domain', 'scale', 'nomenclature']:
    results_df[col] = results_df[col].astype(str).str.strip()

scale_order = ['high_scale', 'low_scale', 'low_neg_scale', 'high_neg_scale']
model_order = ['Qwen8B', 'Qwen14B', 'Qwen32B']  # Custom model order
histories = results_df['History'].unique()
variances = results_df['Variance'].unique()
nomenclatures = results_df['nomenclature'].unique()

def latex_escape(s):
    """Escape LaTeX special characters in a string."""
    if not isinstance(s, str):
        s = str(s)
    s = s.replace('_', r'\_')
    # Add more replacements as needed
    return s

for history, variance in product(histories, variances):
    subset = results_df[
        (results_df['History'] == history) &
        (results_df['Variance'] == variance)
    ]
    if subset.empty:
        print(f"subset {history} {variance} empty!")
        continue

    subset = subset.copy()
    subset['scale'] = pd.Categorical(subset['scale'], categories=scale_order, ordered=True)
    # Sort by Domain, Model (custom order), nomenclature
    subset['Model'] = pd.Categorical(subset['Model'], categories=model_order, ordered=True)
    subset = subset.sort_values(['Domain', 'Model', 'nomenclature'])

    # Get unique domains, models, nomenclatures, and scales in order
    domains = subset['Domain'].unique()
    models = model_order  # Always show all models in this order
    scales = scale_order  # Always show all scales in this order
    noms = nomenclatures

    # Precompute min values for bolding
    # For each domain, for each scale and metric, find the minimum value among (model, nomenclature)
    min_vals = {}
    for domain in domains:
        min_vals[domain] = {}
        domain_subset = subset[subset['Domain'] == domain]
        for scale in scales:
            min_vals[domain][scale] = {}
            for metric in ['step_regret_0_mean', 'step_regret_9_mean', 'cum_regret_9_mean']:
                vals = []
                for model in models:
                    for nom in noms:
                        nom_row = domain_subset[
                            (domain_subset['Model'] == model) &
                            (domain_subset['nomenclature'] == nom) &
                            (domain_subset['scale'] == scale)
                        ]
                        if not nom_row.empty:
                            val = nom_row.iloc[0][metric]
                            if not np.isnan(val):
                                vals.append(val)
                min_vals[domain][scale][metric] = np.min(vals) if vals else None

    # Build LaTeX table header
    header = (
        "\\begin{table}[ht]"
        "\n\\centering"
        "\n\\resizebox{\\textwidth}{!}{%"
        "\n\\renewcommand{\\arraystretch}{1.2}"
        "\n\\begin{tabular}{|l|l|l|" + "|".join(["ccc" for _ in scales]) + "|}"
        "\n\\hline"
        "\n\\textbf{Domain} & \\textbf{Model} & \\textbf{Nomenclature} "
    )
    for scale in scales:
        header += f"& \\multicolumn{{3}}{{c|}}{{\\textbf{{{latex_escape(scale)}}}}} "
    header += "\\\\\n\\hline"
    ## Add triple initial & to account for domain, model, and nomenclature columns
    header += "\n " + " & & & " + " & ".join([" & ".join(["regret@1", "regret@10", "Cum."]) for _ in scales]) + " \\\\"
    header += "\n\\hline"

    # Build table rows: one per domain, model, nomenclature
    rows = ""
    for domain in domains:
        for model in models:
            for nom in noms:
                row_str = f"{latex_escape(domain)} & {latex_escape(model)} & {latex_escape(nom)}"
                for scale in scales:
                    nom_row = subset[
                        (subset['Domain'] == domain) &
                        (subset['Model'] == model) &
                        (subset['nomenclature'] == nom) &
                        (subset['scale'] == scale)
                    ]
                    metrics = ['step_regret_0_mean', 'step_regret_9_mean', 'cum_regret_9_mean']
                    for metric in metrics:
                        if not nom_row.empty:
                            val = nom_row.iloc[0][metric]
                            # Bold if this is the minimum for this domain/scale/metric
                            if min_vals[domain][scale][metric] is not None and val == min_vals[domain][scale][metric]:
                                row_str += f" & \\textbf{{{val:.2f}}}"
                            else:
                                row_str += f" & {val:.2f}"
                        else:
                            row_str += " & --"
                row_str += " \\\\"
                rows += row_str + "\n"
            rows += "\\hline\n"
        rows += "\\hline\n"
    rows += "\\hline\n"

    caption = f"Mean regrets for History={latex_escape(history)}, Variance={latex_escape(variance)}"
    label = f"tab:allmodels_{latex_escape(history)}_{latex_escape(variance)}".replace(" ", "_")

    table = (
        header +
        "\n" +
        rows +
        "\\end{tabular}}\n"  # close tabular and resizebox
        f"\\caption{{{caption}}}\n"
        f"\\label{{{label}}}\n" +
        "\\end{table}"
    )
    print(table)

    # Save to file
    save_path = f"./tables/regret/{history}/all_models/{variance}/table.txt"
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    with open(save_path, "w") as f:
        f.write(table)


\begin{table}[ht]
\centering
\resizebox{\textwidth}{!}{%
\renewcommand{\arraystretch}{1.2}
\begin{tabular}{|l|l|l|ccc|ccc|ccc|ccc|}
\hline
\textbf{Domain} & \textbf{Model} & \textbf{Nomenclature} & \multicolumn{3}{c|}{\textbf{high\_scale}} & \multicolumn{3}{c|}{\textbf{low\_scale}} & \multicolumn{3}{c|}{\textbf{low\_neg\_scale}} & \multicolumn{3}{c|}{\textbf{high\_neg\_scale}} \\
\hline
  & & & regret@1 & regret@10 & Cum. & regret@1 & regret@10 & Cum. & regret@1 & regret@10 & Cum. & regret@1 & regret@10 & Cum. \\
\hline
Bandit & Qwen8B & alphanumeric & 31.25 & \textbf{0.00} & 106.25 & 0.19 & 0.06 & 1.44 & 0.22 & 0.06 & 1.44 & 25.00 & 3.57 & 139.29 \\
Bandit & Qwen8B & sem\_rel\_helpful & \textbf{0.00} & \textbf{0.00} & \textbf{3.12} & \textbf{0.00} & \textbf{0.00} & 0.31 & \textbf{0.00} & 0.07 & 1.21 & \textbf{0.00} & \textbf{0.00} & 134.38 \\
Bandit & Qwen8B & sem\_rel\_mislead & 50.00 & 50.00 & 500.00 & 0.50 & 0.34 & 3.22 & 0.50 & 0.06 & 1.94 & 50.00 & \textbf{0.00} & 196.88 \\
Bandi

### Exploration Count (Need to Edit - how would it work?)

In [9]:
from itertools import product
import numpy as np

# Standardize string columns to avoid matching issues
for col in ['Model', 'History', 'Variance', 'Domain', 'scale', 'nomenclature']:
    results_df[col] = results_df[col].astype(str).str.strip()

scale_order = ['high_scale', 'low_scale', 'low_neg_scale', 'high_neg_scale']
model_order = ['Qwen8B', 'Qwen14B', 'Qwen32B']  # Custom model order
histories = results_df['History'].unique()
variances = results_df['Variance'].unique()
nomenclatures = results_df['nomenclature'].unique()

def latex_escape(s):
    """Escape LaTeX special characters in a string."""
    if not isinstance(s, str):
        s = str(s)
    s = s.replace('_', r'\_')
    # Add more replacements as needed
    return s

for history, variance in product(histories, variances):
    subset = results_df[
        (results_df['History'] == history) &
        (results_df['Variance'] == variance)
    ]
    if subset.empty:
        print(f"subset {history} {variance} empty!")
        continue

    subset = subset.copy()
    subset['scale'] = pd.Categorical(subset['scale'], categories=scale_order, ordered=True)
    # Sort by Domain, Model (custom order), nomenclature
    subset['Model'] = pd.Categorical(subset['Model'], categories=model_order, ordered=True)
    subset = subset.sort_values(['Domain', 'Model', 'nomenclature'])

    # Get unique domains, models, nomenclatures, and scales in order
    domains = subset['Domain'].unique()
    models = model_order  # Always show all models in this order
    scales = scale_order  # Always show all scales in this order
    noms = nomenclatures

    # Precompute max values for bolding (highest value)
    # For each domain, for each scale and metric, find the maximum value among (model, nomenclature)
    max_vals = {}
    for domain in domains:
        max_vals[domain] = {}
        domain_subset = subset[subset['Domain'] == domain]
        for scale in scales:
            max_vals[domain][scale] = {}
            for metric in ['exploration_count_2_mean', 'exploration_count_9_mean']:
                vals = []
                for model in models:
                    for nom in noms:
                        nom_row = domain_subset[
                            (domain_subset['Model'] == model) &
                            (domain_subset['nomenclature'] == nom) &
                            (domain_subset['scale'] == scale)
                        ]
                        if not nom_row.empty:
                            val = nom_row.iloc[0][metric]
                            if not np.isnan(val):
                                vals.append(val)
                max_vals[domain][scale][metric] = np.max(vals) if vals else None

    # Build LaTeX table header
    header = (
        "\\begin{table}[ht]"
        "\n\\centering"
        "\n\\resizebox{\\textwidth}{!}{%"
        "\n\\renewcommand{\\arraystretch}{1.2}"
        "\n\\begin{tabular}{|l|l|l|" + "|".join(["cc" for _ in scales]) + "|}"
        "\n\\hline"
        "\n\\textbf{Domain} & \\textbf{Model} & \\textbf{Nomenclature} "
    )
    for scale in scales:
        header += f"& \\multicolumn{{2}}{{c|}}{{\\textbf{{{latex_escape(scale)}}}}} "
    header += "\\\\\n\\hline"
    ## Add triple initial & to account for domain, model, and nomenclature columns
    header += "\n " + " & & & " + " & ".join([" & ".join(["Turn 3", "Turn 9"]) for _ in scales]) + " \\\\"
    header += "\n\\hline"

    # Build table rows: one per domain, model, nomenclature
    rows = ""
    for domain in domains:
        for model in models:
            for nom in noms:
                row_str = f"{latex_escape(domain)} & {latex_escape(model)} & {latex_escape(nom)}"
                for scale in scales:
                    nom_row = subset[
                        (subset['Domain'] == domain) &
                        (subset['Model'] == model) &
                        (subset['nomenclature'] == nom) &
                        (subset['scale'] == scale)
                    ]
                    metrics = ['exploration_count_2_mean', 'exploration_count_9_mean']
                    for metric in metrics:
                        if not nom_row.empty:
                            val = nom_row.iloc[0][metric]
                            # Bold if this is the maximum for this domain/scale/metric
                            if max_vals[domain][scale][metric] is not None and val == max_vals[domain][scale][metric]:
                                row_str += f" & \\textbf{{{val:.2f}}}"
                            else:
                                row_str += f" & {val:.2f}"
                        else:
                            row_str += " & --"
                row_str += " \\\\"
                rows += row_str + "\n"
            rows += "\\hline\n"
        rows += "\\hline\n"
    rows += "\\hline\n"

    caption = f"Mean exploration count for History={latex_escape(history)}, Variance={latex_escape(variance)}"
    label = f"tab:explorecount_{latex_escape(history)}_{latex_escape(variance)}".replace(" ", "_")

    table = (
        header +
        "\n" +
        rows +
        "\\end{tabular}}\n"  # close tabular and resizebox
        f"\\caption{{{caption}}}\n"
        f"\\label{{{label}}}\n" +
        "\\end{table}"
    )
    print(table)

    # Save to file
    save_path = f"./tables/exploration_count/{history}/all_models/{variance}/table.txt"
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    with open(save_path, "w") as f:
        f.write(table)


\begin{table}[ht]
\centering
\resizebox{\textwidth}{!}{%
\renewcommand{\arraystretch}{1.2}
\begin{tabular}{|l|l|l|cc|cc|cc|cc|}
\hline
\textbf{Domain} & \textbf{Model} & \textbf{Nomenclature} & \multicolumn{2}{c|}{\textbf{high\_scale}} & \multicolumn{2}{c|}{\textbf{low\_scale}} & \multicolumn{2}{c|}{\textbf{low\_neg\_scale}} & \multicolumn{2}{c|}{\textbf{high\_neg\_scale}} \\
\hline
  & & & Turn 3 & Turn 9 & Turn 3 & Turn 9 & Turn 3 & Turn 9 & Turn 3 & Turn 9 \\
\hline
Bandit & Qwen8B & alphanumeric & 2.62 & 2.75 & 2.50 & 2.62 & \textbf{3.00} & \textbf{3.00} & \textbf{3.00} & \textbf{3.00} \\
Bandit & Qwen8B & sem\_rel\_helpful & 1.00 & 1.12 & 1.88 & 1.88 & 2.43 & \textbf{3.00} & 2.50 & \textbf{3.00} \\
Bandit & Qwen8B & sem\_rel\_mislead & 1.00 & 1.00 & 1.75 & 2.12 & 2.62 & 2.88 & \textbf{3.00} & \textbf{3.00} \\
Bandit & Qwen8B & sent\_helpful & 2.38 & 2.38 & 2.50 & 2.50 & 2.75 & \textbf{3.00} & \textbf{3.00} & \textbf{3.00} \\
Bandit & Qwen8B & sent\_mislead & 2.12 & 2.38 & 2.62 & 2